# C1.2 · Red-teaming an agent: designing the campaign

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Security of AI*

Builds on **[C1.1 · The agentic offensive workflow, and containing it](https://spbreed.github.io/cyber-commons/lessons/C1.1.html)**.

| | |
|---|---|
| Tools used | garak, promptfoo, SPIRE, Falco, Llama 3.3, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Run a campaign across the three surfaces and report a rate with its sample size, not an anecdote.

**Why a security engineer needs it.** A red-team result nobody can act on, because "it worked once" is not a rate. The control it builds is: systematic campaigns across all three surfaces, with measured success rates and a criterion agreed before the first payload.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"Can this be jailbroken?" is unfalsifiable and the answer is always yes. Replace it with a number — what fraction of a defined suite reaches a privileged tool — and the conversation becomes one an engineering team can close.

> **At CyberTravels.** One campaign across CyberTravels' three surfaces — the booking note it reads, the delegation it acts under, the refund endpoint it can reach — reporting a rate rather than the one payload that worked.

## 2 · The framework

```
   three surfaces, one scoring method

   injection    what it reads      +--> suite of attacks + BENIGN controls
   identity     who it acts as     +--> run n times
   containment  what it reaches    +--> ASR = reached / attacks
                                   +--> usability = benign that still work

   report both, per surface, with the sample size.
   a defence at 67% ASR and 50% false alarms is worse than nothing.
```

An agent has three attack surfaces, and a red-team engagement has to cover all
three: **injection** (what it reads), **identity** (who it acts as), and
**containment** (what it can reach). What makes the engagement a campaign rather
than a demo is that all three are scored the same way.

So abandon the question "can this chatbot be jailbroken?" — which is
unfalsifiable and always yes — and replace it with one you can put a number on:

> **What fraction of a defined attack suite reaches a privileged tool?**

That is attack success rate (ASR). It is measurable, comparable between builds,
and it goes down when you fix something. Injection is the worked example below
because it is the surface people get wrong most often; the last section runs the
identical scoring across all three.

The suite has to contain two categories that teams usually omit:

- **Keyword-free attacks.** Payloads with none of the vocabulary a filter looks
  for. These are the ones that get through, and they are easy to write.
- **Benign controls.** Ordinary security discussion that *contains* alarming
  words. If your defence flags these, it is not safe, it is unusable — and
  measuring only ASR will never tell you.

## 3 · Where it breaks — reading the keyword row honestly

The keyword filter blocks the two loud attacks and lets all four quiet ones through, so ASR is 0.67. Worse, it fires on two of the four benign cases — ordinary security writing. A defence with 67% ASR *and* a 50% false-alarm rate on legitimate traffic is not a partial win; it is strictly worse than nothing, because it costs trust while providing little.

Provenance blocks everything at 0.00 ASR with no false alarms — which should make you suspicious. A perfect score usually means the suite is not testing the right thing.

## 4 · The same scoring across all three surfaces

Injection was the worked example. Identity and containment are scored with the same two numbers, against the same criterion, and the campaign report is one table — because a defender needs to know which surface buys the most, not which one you found most interesting.

## 5 · The procedure, as a skill

A block rate with no false-alarm rate is half a measurement. The skill runs a fixed suite against each defence, counts what each does to benign security writing, and then delivers the payload through the channel provenance trusts by construction.

### The skill — [`skills/redteam/attack-success-rate-campaign/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/attack-success-rate-campaign/SKILL.md)

```yaml
name: attack-success-rate-campaign
description: >-
  Measure attack success rate for a defence across a case suite, with false
  alarms on benign cases and a bypass delivered through the channel the defence
  trusts. Use when comparing defences, or when a filter's block rate is being
  reported as its effectiveness.
allowed-tools: Read, Grep, Glob
```

# A block rate with no false-alarm rate is half a measurement

Attack success rate is comparable only when the suite is fixed and the benign
cases are run too. A keyword filter looks respectable on ASR and blocks people
writing about security; a provenance control takes ASR to zero and has no false
alarms — until the payload arrives through the channel it trusts.

## When to use this

Comparing defences, accepting a vendor's block rate, or before a control goes in
front of users who write about security for a living.

## Procedure

**1 — Fix the suite before you measure anything.** Attack cases by surface, and
benign cases that look like attacks — a security engineer's own writing, an
incident report quoting a payload. Changing the suite between defences makes the
numbers incomparable, and it is the most common way this is done wrong.

**2 — Run each defence over the whole suite.** Record ASR per surface, not just
overall: a defence that closes injection and does nothing for identity has an
overall number that hides both facts.

**3 — Record false alarms separately.** They are the cost side. A defence with
ASR 0.67 and 2 false alarms in 4 benign cases is not a trade-off anyone would
take if the second number were reported.

**4 — Attack the assumption, not just the defence.** For provenance, deliver the
payload through the principal channel — the one the control trusts by
construction. Every defence has one, and finding it is the point of the
campaign.

**5 — Report the bypass as a property of the design.** "Provenance holds unless
the payload comes from the principal" is the honest claim, and it tells you the
next control rather than discrediting this one.

## Output contract

```json
{
  "suite": {"attack_cases": 0, "benign_cases": 0, "surfaces": ["str"], "frozen": true},
  "defences": [{"name": "str", "asr_overall": 0.0, "asr_by_surface": {"str": 0.0},
                "false_alarms": 0}],
  "bypass": {"defence": "str", "channel": "str", "asr_after": 0.0},
  "claim": "str"
}
```

## Failure modes

- **Changing the suite per defence.** The numbers stop being comparable.
- **Reporting ASR without false alarms.** The cost side is missing.
- **Treating a bypass as a refutation.** It is the boundary of the claim.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/attack-success-rate-campaign/scripts/attack_success_rate_campaign.py
SCRIPT = "skills/redteam/attack-success-rate-campaign/scripts/attack_success_rate_campaign.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

No defence gives ASR 1.00. The keyword filter gives ASR 0.67 with false alarms on 2 of 4 benign security-writing cases. Provenance gives ASR 0.00 with no false alarms — until the payload is delivered through the principal channel, where ASR returns to 1.00. The same two numbers then score all three surfaces in one table.

## Your turn

Run the campaign on all three surfaces against one agent you own, and publish the table rather than the best finding. Start with the list of channels your agent treats as principal-supplied: task descriptions, ticket titles and chat messages usually qualify, and a much wider group can write into them than you expect.

---

**Next → [C1.3 · Attacking evaluation itself](https://spbreed.github.io/cyber-commons/lessons/C1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*